# Отчёт по этапу 4 дипломного проекта

**Тема:** «Разработка интеллектуального Telegram-бота на основе языковой модели для консультирования по нормативной документации в области электроэнергетики (ПУЭ, ПТЭЭП, ГОСТ)» (проект Diplom_AI — RAG-система по нормативным документам).

**Этап 4:** создание прототипа и получение первой точности (для RAG: прототип рабочего контура и метрики Hit@k, MRR/nDCG, генерации и цитирования).

**Студент:** Aleksander Ozyaz  
**Группа:** ____________________  
**Руководитель:** ____________________  
**Репозиторий:** https://github.com/kirag-ozyaz/Diplom_AI

---

**Как пользоваться.** Нажмите *Run All* (Выполнить все). Ноутбук читает только сохранённые файлы результатов из `scripts/stage4_eval/results/` (и `data/`); Milvus, Ollama и GPU не нужны. Корень репозитория определяется автоматически. Повторный прогон оценки (нужны Docker, Milvus, Ollama) — в разделе 6, по умолчанию выключен флагом `RUN_LIVE = False`.

Опыты пронумерованы с 1: опыт 1 — база (`diplom_multimodal`), опыт 2 — префиксы E5 (`diplom_multimodal_e5`), опыт 3 — E5 + реранкер (`diplom_multimodal_e5_rerank`). В файлах проекта те же опыты обозначены 0, 1, 2 (суффиксы JSON: нет / `_e5` / `_rerank`).

## 0. Подготовка: поиск корня проекта и вспомогательные функции

In [ ]:
# Флаг повторного прогона оценки (раздел 6). False — только чтение сохранённых результатов.
RUN_LIVE = False

import json, re
from collections import Counter
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

def find_root(start=None):
    """Корень репозитория: первая папка вверх от текущей, где есть scripts/stage4_eval/results."""
    start = Path(start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "scripts" / "stage4_eval" / "results").is_dir():
            return p
    raise FileNotFoundError("Не найден корень проекта (нет scripts/stage4_eval/results). "
                            "Откройте ноутбук внутри репозитория Diplom_AI.")

ROOT = find_root()
RES = ROOT / "scripts" / "stage4_eval" / "results"
RDIR, GDIR = RES / "retrieval", RES / "generation"
REPORTS = ROOT / "Этапы" / "Reports" / "etap4"
QUESTIONS = ROOT / "data" / "rag_eval_questions" / "eval_questions.json"

EXPERIMENTS = [  # (номер, подпись, суффикс hitk/gen/perf/judge, режим rank)
    (1, "Опыт 1 (база)", "", "base"),
    (2, "Опыт 2 (E5)", "_e5", "e5"),
    (3, "Опыт 3 (реранкер)", "_rerank", "rerank"),
]
LABELS = [e[1] for e in EXPERIMENTS]
COLORS = ["#7f7f7f", "#1f77b4", "#2ca02c"]
plt.rcParams.update({"figure.dpi": 100, "font.size": 10})

def load(p):
    p = Path(p)
    return json.loads(p.read_text(encoding="utf-8-sig")) if p.is_file() else None

def num(v, d=1):
    """Число в русском формате: 81.3 -> '81,3'."""
    if v is None:
        return "—"
    if isinstance(v, (int, float)) and not isinstance(v, bool):
        return f"{v:.{d}f}".replace(".", ",")
    return str(v)

def md_table(header, rows):
    out = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    out += ["| " + " | ".join(str(x) for x in r) + " |" for r in rows]
    display(Markdown("\n".join(out)))

def bar_chart(categories, series, title, ylabel, ymax=None, fmt="{:.1f}", colors=COLORS):
    """Сгруппированная столбчатая диаграмма: series = [(подпись, [значения])]."""
    fig, ax = plt.subplots(figsize=(9, 4))
    x = np.arange(len(categories)); w = 0.8 / len(series)
    for i, (name, vals) in enumerate(series):
        vals = [np.nan if v is None else v for v in vals]
        b = ax.bar(x + (i - (len(series) - 1) / 2) * w, vals, w, label=name,
                   color=colors[i % len(colors)], edgecolor="black", linewidth=0.5)
        for r, v in zip(b, vals):
            if not np.isnan(v):
                ax.text(r.get_x() + r.get_width() / 2, v, fmt.format(v).replace(".", ","),
                        ha="center", va="bottom", fontsize=7)
    ax.set_xticks(x); ax.set_xticklabels(categories)
    if ymax: ax.set_ylim(0, ymax)
    ax.set_ylabel(ylabel); ax.set_title(title); ax.grid(axis="y", ls="--", alpha=.5); ax.set_axisbelow(True)
    ax.legend(fontsize=8, loc="lower right")
    plt.tight_layout(); plt.show()

# Загрузка всех результатов трёх опытов
DATA = {}
for n, label, suf, mode in EXPERIMENTS:
    DATA[n] = {
        "hitk": load(RDIR / f"stage4_hitk_results{suf}.json"),
        "rank": load(RDIR / f"stage4_rank_results_{mode}.json"),
        "gen": load(GDIR / f"stage4_gen_eval_results{suf}.json"),
        "perf": load(GDIR / f"stage4_performance_metrics{suf}.json"),
        "judge": load(GDIR / f"stage4_llm_judge_metrics{suf}.json"),
    }
print("Корень проекта:", ROOT)
for n, label, suf, mode in EXPERIMENTS:
    have = [k for k, v in DATA[n].items() if v is not None]
    miss = [k for k, v in DATA[n].items() if v is None]
    print(f"{label}: есть {have}" + (f"; нет {miss}" if miss else ""))

## 1. Тема, описание задачи

**Задача проекта.** Пользователь задаёт вопрос на русском языке, система находит подходящие пункты нормативного документа и формирует ответ со ссылками на номера пунктов. Используется схема RAG: семантический поиск фрагментов + генерация ответа языковой моделью по найденным фрагментам.

**Задача этапа 4** (по рекомендации куратора объём сужен до ПУЭ):

1. собрать прототип рабочего контура «вопрос → эмбеддинг → Milvus → top-k фрагментов → LLM → ответ со ссылками на пункты ПУЭ»;
2. подготовить контрольный набор вопросов с эталонными пунктами ПУЭ;
3. получить первую точность: Hit@1/3/5 для поиска, MRR и nDCG@k, метрики генерации и цитирования;
4. провести опыты по улучшению поиска и сравнить их на одном наборе вопросов.

Модели предобученные, обучение весов не выполнялось, поэтому графиков loss/accuracy по эпохам нет. «Первой точностью» считается Hit@k — доля вопросов, для которых эталонный пункт ПУЭ попал в первые k найденных фрагментов.

## 2. База данных

**Корпус:** ПУЭ, 7-е издание, главы в DOCX (`data/raw/Нормативная база/ПУЭ/DOCX/`) → Markdown (`data/extracted/`) → чанки JSONL (`data/chunked/`). Документ открытый, конфиденциальных данных нет. ПТЭ (приказ Минэнерго № 811), ПТЭЭП и ГОСТ на этапе 4 в индекс не включались.

Ячейка ниже считает объём базы и наборов вопросов по файлам репозитория.

In [ ]:
chunk_dir = ROOT / "data" / "chunked"
files = sorted(chunk_dir.glob("*.chunked.jsonl")) if chunk_dir.is_dir() else []
if files:
    recs = [json.loads(l) for f in files for l in f.read_text(encoding="utf-8").splitlines() if l.strip()]
    clauses = [r["metadata"].get("Clause") for r in recs if r["metadata"].get("Clause")]
    md_table(["Показатель", "Значение"], [
        ["Файлов DOCX (ПУЭ)", len(list((ROOT / "data/raw/Нормативная база/ПУЭ/DOCX").glob("*.docx")))],
        ["Файлов Markdown", len(list((ROOT / "data/extracted").glob("*.md")))],
        ["Файлов чанков JSONL", len(files)],
        ["Чанков всего", len(recs)],
        ["Чанков с номером пункта (Clause)", f"{len(clauses)} (различных: {len(set(clauses))})"],
        ["Чанков без номера пункта", len(recs) - len(clauses)],
        ["Чанков с таблицами", sum(1 for r in recs if r["metadata"].get("contains_tables"))],
        ["Чанков с изображениями", sum(1 for r in recs if r["metadata"].get("contains_images"))],
    ])
else:
    print("Папка data/chunked не найдена — статистика базы пропущена.")

rows = []
for f in sorted((ROOT / "data" / "rag_eval_questions").glob("*.json")):
    d = load(f)
    n = d.get("n_questions", len(d.get("questions", []))) if isinstance(d, dict) else len(d)
    rows.append([f"`{f.name}`", n, (d.get("registry_range", "") if isinstance(d, dict) else "")])
md_table(["Набор вопросов", "Вопросов", "Диапазон registry_id"], rows)

qs = load(QUESTIONS)["questions"]
SECTION = {q["id"]: q["section"].split(".")[0].replace("Раздел ", "Р.") for q in qs}
cnt = Counter(SECTION.values())
print(f"Итоговый рабочий набор eval_questions.json: {len(qs)} вопросов; по разделам ПУЭ:",
      ", ".join(f"{k} — {cnt[k]}" for k in sorted(cnt)))

## 3. Параметризация данных

**Сегментация.** `src/preprocessing/Create_chunkeds/md_to_chunked_2.py`: уровни Document → Section → Chapter → Paragraph → Clause; один пункт ПУЭ — один чанк; перед текстом добавляется строка контекста «документ. раздел. глава. параграф. Пункт X.Y.Z».

**Индекс Milvus** (по паспортам прогонов): HNSW по `text_vector`, COSINE, M=8, efConstruction=200, поиск ef=64, top-k = 5. Для MRR/nDCG — выдача глубиной 10.

Параметры трёх опытов из конфигурационных файлов:

In [ ]:
cfg_paths = {
    1: ROOT / "config" / "rag_runtime.json",
    2: ROOT / "src/preprocessing/Create_embeddings/E5/rag_runtime_e5.json",
    3: ROOT / "src/preprocessing/Create_embeddings/Rerank/rag_runtime_rerank.json",
}
cfg = {k: load(p) for k, p in cfg_paths.items()}
def g(c, *path):
    for p in path:
        c = (c or {}).get(p)
    return c
rows = []
for name, getter in [
    ("Конфиг", lambda n: f"`{cfg_paths[n].relative_to(ROOT).as_posix()}`"),
    ("Коллекция Milvus", lambda n: g(cfg[n], "vector_db", "collection_name")),
    ("Модель эмбеддингов", lambda n: g(cfg[n], "models", "text_model_name")),
    ("Устройство", lambda n: g(cfg[n], "models", "device_text")),
    ("Префиксы query:/passage:", lambda n: "да" if n > 1 else "нет"),
    ("Реранкер", lambda n: g(cfg[n], "rerank", "model") or "нет"),
    ("Кандидатов из Milvus", lambda n: g(cfg[n], "rerank", "candidates") or g(cfg[n], "query", "default_limit")),
    ("Фрагментов в LLM (top-k)", lambda n: g(cfg[n], "query", "default_limit")),
    ("Записей в коллекции (по JSON Hit@k)", lambda n: g(DATA[n]["hitk"], "collection_num_entities")),
    ("LLM (Ollama)", lambda n: g(cfg[n], "ollama", "model")),
    ("temperature / num_predict", lambda n: f'{g(cfg[n], "ollama", "temperature")} / {g(cfg[n], "ollama", "num_predict")}'),
]:
    rows.append([name] + [getter(n) for n in (1, 2, 3)])
md_table(["Параметр"] + LABELS, rows)

## 4. Архитектура рабочего контура (прототип RAG)

```
Вопрос пользователя
   │  ("query: " + вопрос — в опытах 2 и 3)
   ▼
multilingual-e5-base → вектор 768
   ▼
Milvus (HNSW, COSINE) → top-5 (опыты 1–2) / top-20 (опыт 3)
   ▼
[опыт 3] bge-reranker-v2-m3 → top-5
   ▼
Промпт: системная инструкция + 5 фрагментов ПУЭ + вопрос
   ▼
Ollama qwen2.5:14b → ответ с номерами пунктов ПУЭ
```

| Компонент | Файл |
|---|---|
| Подготовка чанков | `src/preprocessing/Create_chunkeds/md_to_chunked_2.py` |
| Класс RAG (эмбеддинги, Milvus, реранкер) | `src/preprocessing/Create_embeddings/multimodal_rag.py` |
| Загрузка в Milvus, опыты 1 / 2 / 3 | `load_data.py`, `E5/load_data_e5.py`, `Rerank/load_data_rerank.py` |
| Сервис RAG (поиск + ответ) | `src/rag/rag_service.py`, `src/rag/ollama_client.py` |
| Конфигурация | `config/rag_runtime.json` и конфиги опытов 2–3 |
| Инфраструктура | `infra/milvus/`, `infra/docker/Dockerfile.ollama/` |

Telegram-бот (`src/bot/`) к контуру на этапе 4 не подключён.

## 5. Графическое подтверждение (метрики прототипа)

- **Hit@k** (k = 1, 3, 5) — доля вопросов, где в первых k фрагментах есть номер эталонного пункта (`\b{clause}\b`, «Пункт {clause}», «п. {clause}»); один поиск, без LLM.
- **MRR, nDCG@k** — по позиции первого фрагмента с эталонным пунктом в выдаче глубиной 10; при одном релевантном пункте nDCG@k = 1/log2(rank+1).
- **Генерация (live)** — citation_rate, citation_in_answer_rate, groundedness, relevance, время запроса.
- **Performance** — нагрузочный тест Ollama (без Milvus и реранкера).
- **LLM-as-a-Judge** — оценка той же моделью по шаблонному контексту (вспомогательная).

### 5.1. Опыт 1 (база): серия прогонов

Для опыта 1 проведена серия прогонов на последовательных версиях набора вопросов (25, 38, 50, 73, 300, итоговый 268). Источник — журнал `stage4_hitk_runs.jsonl`; подряд идущие прогоны на одном N объединены.

In [ ]:
runs = [json.loads(l) for l in (RDIR / "stage4_hitk_runs.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]
series = []  # группы подряд идущих прогонов с одинаковым N
for r in runs:
    if series and series[-1]["n"] == r["n_questions"]:
        series[-1]["runs"].append(r)
    else:
        series.append({"n": r["n_questions"], "runs": [r]})
rows = []
for i, s in enumerate(series, 1):
    first, last = s["runs"][0], s["runs"][-1]
    m = first["metrics_percent"]
    same = all(x["metrics_percent"] == m for x in s["runs"])
    rows.append([f"1.{i}", s["n"], first["evaluated_at"] + ("" if len(s["runs"]) == 1 else f" … {last['evaluated_at']}"),
                 len(s["runs"]), num(m["hit@1"]) + " %", num(m["hit@3"]) + " %", num(m["hit@5"]) + " %",
                 "да" if same else "нет"])
md_table(["№", "N", "Дата (первый … последний)", "Прогонов", "Hit@1", "Hit@3", "Hit@5", "Повторы совпали"], rows)

fig, ax = plt.subplots(figsize=(9, 4))
xs = [f"N={s['n']}" for s in series]
for k, c in zip(["hit@1", "hit@3", "hit@5"], ["#1f77b4", "#ff7f0e", "#2ca02c"]):
    ax.plot(xs, [s["runs"][0]["metrics_percent"][k] for s in series], marker="o", label=k.replace("hit", "Hit"), color=c)
ax.set_ylim(0, 100); ax.set_ylabel("%"); ax.grid(ls="--", alpha=.5); ax.legend()
ax.set_title("Рисунок 1 — Опыт 1: Hit@k в серии прогонов"); plt.tight_layout(); plt.show()

Падение на 300 вопросах вызвано составом набора (в том числе автоматической разметкой эталонных пунктов), а не поиском. После проверки набор сокращён до 268 вопросов; повторный прогон на 268 вопросах дал те же значения.

### 5.2. Опыт 2 (префиксы E5)

Префиксы `passage:` при индексации и `query:` при поиске, отдельная коллекция `diplom_multimodal_e5`.

### 5.3. Опыт 3 (E5 + реранкер)

Префиксы как в опыте 2; Milvus возвращает 20 кандидатов, `BAAI/bge-reranker-v2-m3` оставляет 5 лучших (для MRR/nDCG — 10).

### 5.4. Сравнение опытов: Hit@k

In [ ]:
rows = []
for k in ["hit@1", "hit@3", "hit@5"]:
    rows.append([k.replace("hit", "Hit")] + [
        f'{num(DATA[n]["hitk"]["metrics_percent"][k])} % ({sum(d["hit"]["@" + k[4:]] for d in DATA[n]["hitk"]["details"])}/{DATA[n]["hitk"]["n_questions"]})'
        for n in (1, 2, 3)])
MISS5 = {n: {d["id"] for d in DATA[n]["hitk"]["details"] if not d["hit"]["@5"]} for n in (1, 2, 3)}
rows.append(["Промахов Hit@5"] + [len(MISS5[n]) for n in (1, 2, 3)])
rows.append(["Дата прогона"] + [DATA[n]["hitk"]["evaluated_at"] for n in (1, 2, 3)])
md_table(["Метрика"] + LABELS, rows)
bar_chart(["Hit@1", "Hit@3", "Hit@5"],
          [(LABELS[n - 1], [DATA[n]["hitk"]["metrics_percent"][k] for k in ("hit@1", "hit@3", "hit@5")]) for n in (1, 2, 3)],
          "Рисунок 2 — Hit@k по трём опытам (N = 268)", "%", ymax=100)
print(f"Опыт 2 исправил {len(MISS5[1] - MISS5[2])} промахов Hit@5 опыта 1 и добавил {len(MISS5[2] - MISS5[1])}; "
      f"опыт 3 исправил {len(MISS5[2] - MISS5[3])} промахов опыта 2 и добавил {len(MISS5[3] - MISS5[2])}; "
      f"не найдено ни в одном опыте: {len(MISS5[1] & MISS5[2] & MISS5[3])}.")

Hit@5 по разделам ПУЭ:

In [ ]:
secs = sorted(set(SECTION.values()))
tab = {}
for n in (1, 2, 3):
    tot, hit = Counter(), Counter()
    for d in DATA[n]["hitk"]["details"]:
        s = SECTION[d["id"]]; tot[s] += 1; hit[s] += d["hit"]["@5"]
    tab[n] = {s: 100 * hit[s] / tot[s] for s in secs}
md_table(["Раздел", "Вопросов"] + LABELS, [[s, cnt[s]] + [num(tab[n][s]) + " %" for n in (1, 2, 3)] for s in secs])
bar_chart(secs, [(LABELS[n - 1], [tab[n][s] for s in secs]) for n in (1, 2, 3)],
          "Рисунок 3 — Hit@5 по разделам ПУЭ", "%", ymax=110, fmt="{:.0f}")

### 5.5. Сравнение опытов: MRR и nDCG@k (глубина 10)

In [ ]:
RANK_KEYS = ["hit@10", "mrr", "mrr@3", "mrr@5", "ndcg@3", "ndcg@5", "ndcg@10"]
RNAME = {"hit@10": "Hit@10", "mrr": "MRR", "mrr@3": "MRR@3", "mrr@5": "MRR@5", "ndcg@3": "nDCG@3", "ndcg@5": "nDCG@5", "ndcg@10": "nDCG@10"}
if all(DATA[n]["rank"] for n in (1, 2, 3)):
    rows = [[RNAME[k]] + [num(DATA[n]["rank"]["metrics"][k], 3) for n in (1, 2, 3)] for k in RANK_KEYS]
    rows.append(["Дата прогона"] + [DATA[n]["rank"]["evaluated_at"] for n in (1, 2, 3)])
    md_table(["Метрика"] + LABELS, rows)
    for n in (1, 2, 3):  # проверка: Hit@1/3/5 в rank-прогоне = основной прогон
        for k in ("hit@1", "hit@3", "hit@5"):
            a, b = DATA[n]["hitk"]["metrics_percent"][k], round(DATA[n]["rank"]["metrics"][k] * 100, 1)
            if abs(a - b) > 0.05:
                print(f"! {LABELS[n-1]}: {k} {a} ≠ {b}")
    bar_chart([RNAME[k] for k in RANK_KEYS], [(LABELS[n - 1], [DATA[n]["rank"]["metrics"][k] for k in RANK_KEYS]) for n in (1, 2, 3)],
              "Рисунок 4 — MRR и nDCG@k по опытам", "0–1", ymax=1.05, fmt="{:.2f}")
else:
    print("Нет stage4_rank_results_*.json — запустите run_rank_all.ps1 (раздел 6).")

### 5.6. Сравнение опытов: генерация и цитирование

Если файл генерации опыта посчитан не на всех вопросах (`partial_run` или N ≠ 268), его значения не используются: для опыта 1 подставляются значения из паспорта полного прогона (`Etap4-hitk-otchet-N268.md`), источник указан в таблице.

In [ ]:
GEN_KEYS = ["citation_rate", "citation_in_answer_rate", "citation_in_context_rate", "groundedness_avg", "relevance_avg"]

def gen_from_passport():
    """Значения генерации опыта 1 из паспорта полного прогона (если JSON перезаписан неполным прогоном)."""
    p = REPORTS / "Etap4-hitk-otchet-N268.md"
    if not p.is_file():
        return None
    text = p.read_text(encoding="utf-8")
    out = {}
    for k in GEN_KEYS:
        m = re.search(rf"\|\s*`{k}`\s*\|[^|]*\|\s*\*\*([\d,]+)", text)
        if m:
            out[k] = float(m.group(1).replace(",", "."))
    svod = REPORTS / "Etap4-hitk-svod.md"
    if svod.is_file():
        m = re.search(r"duration_avg_sec \(generation\)\s*\|\s*~?([\d,]+)", svod.read_text(encoding="utf-8"))
        if m:
            out["duration_avg_sec"] = float(m.group(1).replace(",", "."))
    return out or None

GEN, GEN_SRC = {}, {}
for n, label, suf, mode in EXPERIMENTS:
    g_ = DATA[n]["gen"]; N = DATA[n]["hitk"]["n_questions"]
    m = (g_ or {}).get("metrics", {})
    if g_ is not None and not m.get("partial_run") and m.get("n_questions") == N:
        GEN[n] = dict(m); GEN[n]["errors"] = sum(1 for d in g_["details"] if d.get("error"))
        GEN_SRC[n] = f"stage4_gen_eval_results{suf}.json ({g_['evaluated_at']})"
    else:
        pp = gen_from_passport() if n == 1 else None
        GEN[n] = pp or {}
        GEN_SRC[n] = ("паспорт Etap4-hitk-otchet-N268.md; JSON перезаписан неполным прогоном "
                      f"({m.get('n_questions')} из {N}, {g_.get('evaluated_at') if g_ else '—'})") if pp else "нет полного прогона"
rows = [[k] + [num(GEN[n].get(k), 1 if k.endswith("rate") else 3) for n in (1, 2, 3)] for k in GEN_KEYS]
rows.append(["Среднее время запроса, с"] + [("~" if "паспорт" in GEN_SRC[n] else "") + num(GEN[n].get("duration_avg_sec"), 2) for n in (1, 2, 3)])
rows.append(["Ошибок live"] + [GEN[n].get("errors", "нет данных") for n in (1, 2, 3)])
rows.append(["Источник"] + [GEN_SRC[n] for n in (1, 2, 3)])
md_table(["Метрика"] + LABELS, rows)
bar_chart(["citation_rate", "citation_in_answer_rate"],
          [(LABELS[n - 1], [GEN[n].get("citation_rate"), GEN[n].get("citation_in_answer_rate")]) for n in (1, 2, 3)],
          "Рисунок 5 — Цитирование эталонного пункта", "%", ymax=100)
bar_chart(["groundedness_avg", "relevance_avg"],
          [(LABELS[n - 1], [GEN[n].get("groundedness_avg"), GEN[n].get("relevance_avg")]) for n in (1, 2, 3)],
          "Рисунок 6 — Groundedness и relevance", "0–1", ymax=1.05, fmt="{:.3f}")

### 5.7. Время, производительность и LLM-as-a-Judge

In [ ]:
def durs(d):
    return [x.get("duration_sec") for x in (d or {}).get("details", []) if x.get("duration_sec") is not None]
TIME = {}
for n in (1, 2, 3):
    xs = durs(DATA[n]["rank"])
    TIME[n] = {"rag": GEN[n].get("duration_avg_sec"),
               "retr": sum(xs) / len(xs) if xs else None,
               "retr_med": float(np.median(xs)) if xs else None,
               "first": xs[0] if xs else None}
md_table(["Время"] + LABELS, [
    ["Полный RAG, с/вопрос (generation live, среднее)"] + [("~" if "паспорт" in GEN_SRC[n] else "") + num(TIME[n]["rag"], 2) for n in (1, 2, 3)],
    ["Только поиск, с/вопрос (rank-прогон, среднее)"] + [num(TIME[n]["retr"], 3) for n in (1, 2, 3)],
    ["Только поиск, с/вопрос (медиана)"] + [num(TIME[n]["retr_med"], 3) for n in (1, 2, 3)],
    ["Первый запрос (холодный старт), с"] + [num(TIME[n]["first"], 2) for n in (1, 2, 3)],
])
bar_chart(["Полный RAG", "Только поиск"], [(LABELS[n - 1], [TIME[n]["rag"], TIME[n]["retr"]]) for n in (1, 2, 3)],
          "Рисунок 7 — Среднее время на вопрос, с", "с", fmt="{:.2f}")

PK = [("ttft_avg_sec", "TTFT avg, с"), ("latency_avg_sec", "Latency avg, с"), ("latency_p95_sec", "Latency p95, с"),
      ("latency_p99_sec", "Latency p99, с"), ("tps_avg", "TPS, ток/с")]
rows = [[name] + [num(DATA[n]["perf"]["metrics"][k], 2) for n in (1, 2, 3)] for k, name in PK]
rows += [["RPS"] + [num(DATA[n]["perf"]["rps"], 2) for n in (1, 2, 3)],
         ["Доступность"] + [f'{num(DATA[n]["perf"]["availability_pct"])} % ({DATA[n]["perf"]["successful_requests"]}/{DATA[n]["perf"]["total_requests"]})' for n in (1, 2, 3)]]
md_table(["Нагрузочный тест Ollama"] + LABELS, rows)
bar_chart(["TTFT", "Latency avg", "Latency p95", "Latency p99"],
          [(LABELS[n - 1], [DATA[n]["perf"]["metrics"][k] for k, _ in PK[:4]]) for n in (1, 2, 3)],
          "Рисунок 8 — Нагрузочный тест Ollama, с", "с", fmt="{:.1f}")

JK = [("faithfulness_avg", "Faithfulness"), ("answer_relevancy_avg", "Answer relevancy"), ("context_recall_avg", "Context recall")]
JFAIL = {n: [d["id"] for d in DATA[n]["judge"]["details"] if "Ошибка вызова" in str((d.get("scores") or {}).get("reasoning", ""))] for n in (1, 2, 3)}
rows = [[name] + [num(DATA[n]["judge"]["summary_metrics"][k], 4) for n in (1, 2, 3)] for k, name in JK]
rows.append(["Оценено без сбоя"] + [f'{DATA[n]["judge"]["sample_count"] - len(JFAIL[n])}/{DATA[n]["judge"]["sample_count"]}'
                                     + (f" (сбой id {JFAIL[n]})" if JFAIL[n] else "") for n in (1, 2, 3)])
md_table(["LLM-as-a-Judge"] + LABELS, rows)
bar_chart([name for _, name in JK], [(LABELS[n - 1], [DATA[n]["judge"]["summary_metrics"][k] for k, _ in JK]) for n in (1, 2, 3)],
          "Рисунок 9 — LLM-as-a-Judge", "0–1", ymax=0.7, fmt="{:.3f}")

Нагрузочный тест не использует поиск, поэтому различия между опытами в нём — разброс очереди GPU. Судье подаётся одинаковый шаблонный контекст, поэтому его оценки почти не различаются.

## 6. Ноутбук или .py файл (прототип)

Этот ноутбук (`Этапы/Reports/etap4/Etap4_final.ipynb`) — итоговый отчёт этапа: только чтение результатов, вывод появляется после запуска ячеек. Оценку выполняют скрипты `scripts/stage4_eval/`:

| Файл | Назначение |
|---|---|
| `retrieval/eval_retrieval_hitk.py`, `_e5.py`, `_rerank.py` | Hit@k, опыты 1 / 2 / 3 |
| `retrieval/mrr_ndcg/eval_retrieval_rank.py` | MRR, MRR@k, nDCG@k, Hit@10 (`--mode base/e5/rerank`) |
| `retrieval/mrr_ndcg/rank_metrics.py` | сводная таблица `rank_metrics_comparison.md/.csv` |
| `generation/eval_rag_generation.py` | генерация live: citation, groundedness, relevance, время |
| `generation/eval_rag_performance.py` | нагрузочный тест Ollama |
| `generation/eval_rag_llm_judge.py` | LLM-as-a-Judge |
| `reports/build_stage4_summary.py` | автоматическая сводка всех метрик трёх опытов (md/csv/json), если скрипт добавлен в проект |
| `demo/demo_stage4_milvus_search.py` | демо поиска в Milvus |
| `run_all.ps1`, `run_all_e5.ps1`, `run_all_rerank.ps1` | полный прогон опыта 1 / 2 / 3 |
| `run_rank_all.ps1` | MRR/nDCG для трёх опытов |

Ячейка ниже проверяет наличие скриптов. Следующая — повторный прогон (Docker, Milvus, Ollama, Windows/PowerShell); выполняется только при `RUN_LIVE = True` (ячейка раздела 0) и перезаписывает файлы результатов.

In [ ]:
SCRIPTS = ["retrieval/eval_retrieval_hitk.py", "retrieval/eval_retrieval_hitk_e5.py", "retrieval/eval_retrieval_hitk_rerank.py",
           "retrieval/mrr_ndcg/eval_retrieval_rank.py", "retrieval/mrr_ndcg/rank_metrics.py",
           "generation/eval_rag_generation.py", "generation/eval_rag_performance.py", "generation/eval_rag_llm_judge.py",
           "reports/build_stage4_summary.py", "demo/demo_stage4_milvus_search.py",
           "run_all.ps1", "run_all_e5.ps1", "run_all_rerank.ps1", "run_rank_all.ps1"]
md_table(["Скрипт", "Есть"], [[f"`scripts/stage4_eval/{s}`", "да" if (ROOT / "scripts/stage4_eval" / s).is_file() else "НЕТ"] for s in SCRIPTS])
lib = ROOT / "scripts" / "lib"
print("scripts/lib (нужна скриптам оценки):", "есть" if lib.is_dir() else "НЕТ — скрипты оценки не запустятся (в .gitignore строка lib/)")

In [ ]:
# ПОВТОРНЫЙ ПРОГОН ОЦЕНКИ — только при RUN_LIVE = True (нужны Docker, Milvus, Ollama, scripts/lib).
import subprocess
if not RUN_LIVE:
    print("RUN_LIVE = False — повторный прогон пропущен, используются сохранённые результаты.")
else:
    steps = [
        ["powershell", "-ExecutionPolicy", "Bypass", "-File", r"scripts\stage4_eval\run_all.ps1"],
        ["powershell", "-ExecutionPolicy", "Bypass", "-File", r"scripts\stage4_eval\run_all_e5.ps1"],
        ["powershell", "-ExecutionPolicy", "Bypass", "-File", r"scripts\stage4_eval\run_all_rerank.ps1"],
        ["powershell", "-ExecutionPolicy", "Bypass", "-File", r"scripts\stage4_eval\run_rank_all.ps1", "-SkipInfra"],
    ]
    for cmd in steps:
        print(">", " ".join(cmd))
        rc = subprocess.run(cmd, cwd=str(ROOT)).returncode
        if rc != 0:
            raise RuntimeError(f"Шаг завершился с кодом {rc}")
    print("Готово. Перезапустите ноутбук (Run All), чтобы перечитать результаты.")

## 7. Выводы

Числа в выводах подставляются из результатов выше.

In [ ]:
h = {n: DATA[n]["hitk"]["metrics_percent"] for n in (1, 2, 3)}
r = {n: (DATA[n]["rank"] or {}).get("metrics", {}) for n in (1, 2, 3)}
min5 = min(s["runs"][0]["metrics_percent"]["hit@5"] for s in series)
text = f"""
1. Прототип рабочего контура RAG по ПУЭ собран: поиск в Milvus, генерация в Ollama; доступность LLM в нагрузочном тесте {num(DATA[3]["perf"]["availability_pct"])} %, ошибок live-генерации в опытах 2 и 3: {GEN[2].get("errors")} и {GEN[3].get("errors")}.
2. Серия прогонов опыта 1 показала зависимость результата от состава набора вопросов (Hit@5 от {num(min5)} % до {num(h[1]["hit@5"])} %); итоговый набор — 268 вопросов.
3. Опыт 1 (база): Hit@1 {num(h[1]["hit@1"])} %, Hit@5 {num(h[1]["hit@5"])} %.
4. Опыт 2 (префиксы E5): Hit@1 {num(h[2]["hit@1"])} %, Hit@5 {num(h[2]["hit@5"])} % — небольшой прирост без роста времени ответа.
5. Опыт 3 (реранкер): Hit@1 {num(h[3]["hit@1"])} %, Hit@5 {num(h[3]["hit@5"])} %, цитирование {num(GEN[3].get("citation_rate"))} %, relevance {num(GEN[3].get("relevance_avg"), 3)}; время запроса {num(GEN[3].get("duration_avg_sec"), 2)} с против {num(GEN[2].get("duration_avg_sec"), 2)} с в опыте 2; поиск с реранкером ~{num(TIME[3]["retr"], 2)} с против ~{num(TIME[2]["retr"], 2)} с.
6. Ранговые метрики (глубина 10): MRR {num(r[1].get("mrr"), 3)} → {num(r[2].get("mrr"), 3)} → {num(r[3].get("mrr"), 3)}, nDCG@5 {num(r[1].get("ndcg@5"), 3)} → {num(r[2].get("ndcg@5"), 3)} → {num(r[3].get("ndcg@5"), 3)}.
7. Качество ответа определяется поиском: citation_rate совпадает с Hit@5; номер пункта явно назван в ответе в {num(GEN[3].get("citation_in_answer_rate"))} % случаев (опыт 3).
8. Для дальнейшей работы выбрана конфигурация опыта 3.

**Ограничения:** {len(MISS5[3])} вопросов не найдены в top-5 в опыте 3, {len(MISS5[1] & MISS5[2] & MISS5[3])} — ни в одном опыте; Hit@k засчитывает попадание только по номеру пункта; LLM-judge оценивает шаблонный ответ, сбои судьи: {"; ".join(f"{LABELS[n-1]} — id {', '.join(map(str, JFAIL[n]))}" for n in (1, 2, 3) if JFAIL[n]) or "нет"}; нагрузочный тест не измеряет задержку полного RAG; генерация опыта 1 — {GEN_SRC[1]}.
"""
display(Markdown(text))

## 8. План дальнейшей работы

1. Подключить RAG (конфигурация опыта 3) к Telegram-боту (`src/bot/`).
2. Повторить полный прогон генерации опыта 1 (`run_generation.ps1 -Mode live`); пробные прогоны с `--limit` писать в отдельный файл.
3. Вернуть в репозиторий `scripts/lib` (исключение `!scripts/lib/` в `.gitignore`).
4. Разобрать промахи опыта 3 и общие промахи трёх опытов; исправить разметку набора вопросов.
5. Доработать промпт, чтобы LLM явно указывал номер пункта в ответе.
6. Перевести LLM-judge на фактические ответы и контекст из прогона генерации.
7. Измерять задержку полного контура RAG; собрать бот, Milvus и Ollama в один Docker Compose (этап 5).
8. По возможности добавить ПТЭЭП.

## Приложения

**Соответствие нумерации:** опыт 1 = опыт 0 в файлах проекта, опыт 2 = опыт 1 (E5), опыт 3 = опыт 2 (реранкер).

| Что | Файл |
|---|---|
| Набор вопросов (268) | `data/rag_eval_questions/eval_questions.json` |
| Hit@k | `scripts/stage4_eval/results/retrieval/stage4_hitk_results{,_e5,_rerank}.json` |
| История прогонов опыта 1 | `scripts/stage4_eval/results/retrieval/stage4_hitk_runs.jsonl` |
| MRR/nDCG | `scripts/stage4_eval/results/retrieval/stage4_rank_results_{base,e5,rerank}.json` |
| Генерация, performance, judge | `scripts/stage4_eval/results/generation/` (без суффикса, `_e5`, `_rerank`) |
| Паспорта прогонов | `Этапы/Reports/etap4/Etap4-hitk-otchet-N268.md`, `Etap4-opyt1-e5-N268.md`, `Etap4-opyt2-rerank-N268.md` |